# Hannah - CNN from scratch experiment

**Owner:** Hannah only.

This notebook implements a custom CNN for multi-label movie poster genre classification. It does not use an ImageNet backbone, pretrained weights, ResNet, VGG, EfficientNet, MobileNet, DenseNet, or any other pretrained model.

## Colab + Drive Runtime Setup

To work on this notebook in Colab, put the whole `ieor142b` repo in Google Drive, not just this notebook. Colab cannot read files from your Mac directly, so Drive should contain `experimentation/hannah.ipynb`, `experimentation/shared.py`, `cleaned/MovieGenre_clean_with_images_full.csv`, `splits_cleaned/`, and `cleaned/downloaded_posters/`.

Open the notebook from Colab with **File > Open notebook > Google Drive** after the repo syncs to Drive. Run the setup cell below first. It mounts Drive, copies the repo to `/content/ieor142b_runtime`, and copies poster images to local Colab disk once per runtime so training does not stream thousands of JPGs from Drive.


In [ ]:
# Run this first in Google Colab. It is safe to skip locally.
from pathlib import Path
import os
import shutil

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

USE_RUNTIME_COPY = True
FORCE_REFRESH = False  # Set True once if you need a clean /content copy.
SYNC_MODE = "code_only"  # "code_only" is fastest after the first successful copy.

if IN_COLAB:
    drive.mount("/content/drive")

    DRIVE_REPO = Path("/content/drive/MyDrive/ieor142b")
    if not DRIVE_REPO.exists():
        print(f"Default DRIVE_REPO not found: {DRIVE_REPO}")
        print("Searching MyDrive for an ieor142b repo with the strict cleaned CSV...")
        candidates = []
        for shared_file in Path("/content/drive/MyDrive").rglob("shared.py"):
            if shared_file.parts[-2] != "experimentation":
                continue
            root = shared_file.parent.parent
            if (root / "cleaned" / "MovieGenre_clean_with_images_full.csv").is_file():
                candidates.append(root)
        if not candidates:
            raise RuntimeError("Could not find ieor142b in Drive. Upload/sync the repo folder, then rerun this cell.")
        DRIVE_REPO = candidates[0]
        print("Using discovered DRIVE_REPO:", DRIVE_REPO)

    RUNTIME_REPO = Path("/content/ieor142b_runtime")

    def copy_item(src: Path, dst: Path) -> None:
        if not src.exists():
            return
        if src.is_dir():
            if dst.exists() or dst.is_symlink():
                if dst.is_symlink():
                    dst.unlink()
                else:
                    shutil.rmtree(dst)
            shutil.copytree(src, dst)
        else:
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src, dst)

    code_and_metadata_paths = [
        "experimentation",
        "scripts",
        "results",
        "splits_cleaned",
        "README.md",
        "TEAM_DATA_SETUP.md",
        "cleaned/MovieGenre_clean_with_images_full.csv",
        "cleaned/MovieGenre_clean_with_images_full_manifest.json",
        "cleaned/README.md",
    ]

    if USE_RUNTIME_COPY:
        if FORCE_REFRESH and RUNTIME_REPO.exists():
            shutil.rmtree(RUNTIME_REPO)

        first_runtime_copy = not RUNTIME_REPO.exists()
        RUNTIME_REPO.mkdir(parents=True, exist_ok=True)

        for rel in code_and_metadata_paths:
            copy_item(DRIVE_REPO / rel, RUNTIME_REPO / rel)

        drive_images = DRIVE_REPO / "cleaned" / "downloaded_posters"
        runtime_images = RUNTIME_REPO / "cleaned" / "downloaded_posters"
        runtime_images.parent.mkdir(parents=True, exist_ok=True)

        if first_runtime_copy or FORCE_REFRESH or not runtime_images.exists():
            if runtime_images.exists() or runtime_images.is_symlink():
                if runtime_images.is_symlink():
                    runtime_images.unlink()
                else:
                    shutil.rmtree(runtime_images)
            if drive_images.exists():
                print("Copying poster images from Drive to local /content. This is the slow step, but only once per runtime.")
                shutil.copytree(drive_images, runtime_images)
                print("Runtime poster count:", sum(1 for _ in runtime_images.glob("*.jpg")))
            else:
                print("WARNING: Drive poster folder missing. Run scripts/fetch_cleaned_images.py before full training.")
        else:
            print("Reusing local runtime poster cache:", runtime_images)

        ACTIVE_REPO = RUNTIME_REPO
    else:
        ACTIVE_REPO = DRIVE_REPO

    os.chdir(str(ACTIVE_REPO))
    os.environ["IEOR142B_ROOT"] = str(ACTIVE_REPO)
    os.environ["IEOR142B_DRIVE_REPO"] = str(DRIVE_REPO)

    !pip -q install scikit-learn pandas pillow matplotlib

    print("Active repo root:", ACTIVE_REPO)
    print("Drive repo root:", DRIVE_REPO)
    print("shared.py:", (ACTIVE_REPO / "experimentation" / "shared.py").is_file())
    print("strict csv:", (ACTIVE_REPO / "cleaned" / "MovieGenre_clean_with_images_full.csv").is_file())
    print("strict train split:", (ACTIVE_REPO / "splits_cleaned" / "train_rows.csv").is_file())
    print("poster dir:", ACTIVE_REPO / "cleaned" / "downloaded_posters")
else:
    print("Not running in Colab; using local files.")


In [ ]:
import torch

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
else:
    print("mps available:", torch.backends.mps.is_available())


In [ ]:
from __future__ import annotations

import copy
import json
import os
import shutil
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import transforms


def find_project_root() -> Path:
    root_override = os.environ.get("IEOR142B_ROOT")
    candidates = []
    if root_override:
        candidates.append(Path(root_override).expanduser().resolve())

    here = Path.cwd().resolve()
    candidates.extend([here, *here.parents])
    candidates.extend([
        Path("/content/ieor142b_runtime"),
        Path("/content/drive/MyDrive/ieor142b"),
    ])

    seen = set()
    for base in candidates:
        if base in seen:
            continue
        seen.add(base)
        if (
            (base / "experimentation" / "shared.py").is_file()
            and (base / "cleaned" / "MovieGenre_clean_with_images_full.csv").is_file()
        ):
            return base

    raise RuntimeError(
        "Could not find ieor142b with strict cleaned data. In Colab, run the Drive/runtime setup cell first."
    )


ROOT = find_project_root()
sys.path.insert(0, str(ROOT / "experimentation"))

from shared import STRICT_BASELINE, STRICT_POSTER_DIR, load_strict_split_dataframes, set_seed, save_metrics_json

set_seed(STRICT_BASELINE["seed"])
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print("Project root:", ROOT)
print("Using device:", DEVICE)


In [ ]:
train_df, val_df, test_df, mlb = load_strict_split_dataframes()
num_genres = len(mlb.classes_)


def resolve_poster_dir(base_dir: Path) -> Path:
    candidates = [base_dir, ROOT / base_dir, ROOT / "cleaned" / "downloaded_posters"]
    for candidate in candidates:
        candidate = Path(candidate)
        if candidate.is_dir() and any(candidate.glob("*.jpg")):
            return candidate
    raise FileNotFoundError(
        f"Could not find strict poster JPGs. Expected them under {ROOT / 'cleaned' / 'downloaded_posters'}."
    )


poster_dir = resolve_poster_dir(STRICT_POSTER_DIR)

print("Train / Val / Test:", len(train_df), len(val_df), len(test_df))
print("Number of genres:", num_genres)
print("Poster directory:", poster_dir)


In [ ]:
# Data and image diagnostics: run before training to catch path or imbalance problems.
def label_lists(df: pd.DataFrame) -> list[list[str]]:
    return df["Genre"].fillna("").str.split("|").tolist()


def label_matrix(df: pd.DataFrame) -> np.ndarray:
    return mlb.transform(label_lists(df))


def summarize_split(name: str, df: pd.DataFrame) -> dict:
    y = label_matrix(df)
    labels_per_sample = y.sum(axis=1)
    return {
        "split": name,
        "rows": int(len(df)),
        "avg_labels_per_sample": float(labels_per_sample.mean()),
        "label_density": float(y.mean()),
        "empty_label_rows": int((labels_per_sample == 0).sum()),
    }


def poster_cache_report(df: pd.DataFrame, poster_dir: Path, sample_n: int = 20) -> dict:
    imdb_ids = pd.to_numeric(df["imdbId"], errors="coerce").dropna().astype(int).tolist()
    present = sum((poster_dir / f"{imdb_id}.jpg").is_file() for imdb_id in imdb_ids)
    sample_missing = [
        str(poster_dir / f"{imdb_id}.jpg")
        for imdb_id in imdb_ids[:sample_n]
        if not (poster_dir / f"{imdb_id}.jpg").is_file()
    ]
    return {
        "poster_dir": str(poster_dir),
        "jpg_files_in_dir": int(sum(1 for _ in poster_dir.glob("*.jpg"))),
        "rows_with_matching_jpg": int(present),
        "rows_checked": int(len(imdb_ids)),
        "coverage": float(present / max(1, len(imdb_ids))),
        "sample_missing_paths": sample_missing[:5],
    }


print("Split summaries:")
for split_name, split_df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(summarize_split(split_name, split_df))

print("\nPoster cache report:")
poster_report = poster_cache_report(train_df, poster_dir)
print(poster_report)

class_counts = pd.Series(label_matrix(train_df).sum(axis=0), index=mlb.classes_).sort_values(ascending=False)
print("\nMost common train genres:")
print(class_counts.head(10).to_string())
print("\nLeast common train genres:")
print(class_counts.tail(10).to_string())


## Custom CNN Design

- **Architecture:** Four scratch convolutional blocks using `Conv2d`, `BatchNorm2d`, `ReLU`, `MaxPool2d`, and `Dropout2d`, followed by adaptive average pooling, `Flatten`, `Linear`, `BatchNorm1d`, `ReLU`, `Dropout`, and a final `Linear` layer with one logit per genre.
- **Loss:** Weighted `BCEWithLogitsLoss` is the default because the strict dataset is highly multi-label and imbalanced; focal loss remains available as an option.
- **Metrics:** Reports sample-averaged precision, recall, F1, exact-match accuracy, predicted labels per movie, true labels per movie, and probability summaries.
- **Thresholding:** Training logs still show metrics at the configured default threshold, then a validation sweep picks the final threshold used for test evaluation.
- **Data and splits:** Reuses `load_strict_split_dataframes()` from `experimentation/shared.py`, which loads the strict cleaned train/validation/test CSVs from `splits_cleaned/` and fits labels on the full strict dataframe first.
- **Difference from baseline:** Other baselines may use pretrained backbones; this notebook uses only randomly initialized layers written from scratch. Poster files come from `cleaned/downloaded_posters/`, and the Colab setup cell copies them to local `/content` storage for faster training I/O.
- **Run behavior:** The smoke-test cell trains on a tiny subset by default. Full training, metrics JSON writing, history JSON writing, and checkpoint writing are opt-in.


In [ ]:
class HannahConfig:
    IMG_SIZE = STRICT_BASELINE["img_size"]
    BATCH_SIZE = 32
    NUM_EPOCHS = 12
    LR = 3e-4
    WEIGHT_DECAY = 1e-4
    DROPOUT = 0.35
    PATIENCE = 6
    MIN_VAL_LOSS_DELTA = 1e-4
    LR_FACTOR = 0.3
    LR_PATIENCE = 2
    MIN_LR = 1e-6
    NUM_WORKERS = 2 if torch.cuda.is_available() else 0
    LOSS_NAME = "weighted_bce"  # "weighted_bce" or "focal"
    DEFAULT_THRESHOLD = STRICT_BASELINE["metric_threshold"]
    THRESHOLD_SWEEP = [round(x / 100, 2) for x in range(5, 81, 5)]
    USE_AMP = True
    GRAD_CLIP_NORM = 1.0
    SMOKE_TRAIN_SAMPLES = 8
    SMOKE_VAL_SAMPLES = 8
    SMOKE_BATCH_SIZE = 4


class MoviePosterDataset(Dataset):
    def __init__(self, df: pd.DataFrame, mlb, poster_dir: Path, transform=None):
        self.df = df.reset_index(drop=True)
        self.mlb = mlb
        self.poster_dir = Path(poster_dir)
        self.transform = transform
        self.bad_images = 0

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        imdb_id = str(int(row["imdbId"]))
        img_path = self.poster_dir / f"{imdb_id}.jpg"

        try:
            img = Image.open(img_path).convert("RGB")
        except (FileNotFoundError, OSError):
            self.bad_images += 1
            img = Image.new("RGB", (HannahConfig.IMG_SIZE, HannahConfig.IMG_SIZE), (128, 128, 128))

        if self.transform is not None:
            img = self.transform(img)

        genres = row["Genre"].split("|")
        label = torch.tensor(self.mlb.transform([genres])[0], dtype=torch.float32)
        return img, label


def get_transforms():
    mean = [0.485, 0.456, 0.406]
    std = [0.229, 0.224, 0.225]

    train_tf = transforms.Compose([
        transforms.Resize((HannahConfig.IMG_SIZE + 32, HannahConfig.IMG_SIZE + 32)),
        transforms.RandomCrop(HannahConfig.IMG_SIZE),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])
    eval_tf = transforms.Compose([
        transforms.Resize((HannahConfig.IMG_SIZE, HannahConfig.IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])
    return train_tf, eval_tf


def make_loaders(batch_size: int = HannahConfig.BATCH_SIZE, smoke: bool = False):
    train_tf, eval_tf = get_transforms()
    train_ds = MoviePosterDataset(train_df, mlb, poster_dir, train_tf)
    val_ds = MoviePosterDataset(val_df, mlb, poster_dir, eval_tf)
    test_ds = MoviePosterDataset(test_df, mlb, poster_dir, eval_tf)

    if smoke:
        train_ds = Subset(train_ds, range(min(HannahConfig.SMOKE_TRAIN_SAMPLES, len(train_ds))))
        val_ds = Subset(val_ds, range(min(HannahConfig.SMOKE_VAL_SAMPLES, len(val_ds))))
        test_ds = Subset(test_ds, range(min(HannahConfig.SMOKE_VAL_SAMPLES, len(test_ds))))

    loader_kwargs = {
        "num_workers": HannahConfig.NUM_WORKERS,
        "pin_memory": torch.cuda.is_available(),
    }
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, **loader_kwargs)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, **loader_kwargs)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, **loader_kwargs)
    return train_loader, val_loader, test_loader


In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, alpha: float = 0.25, gamma: float = 2.0, reduction: str = "mean"):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        bce_loss = nn.functional.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        probs = torch.sigmoid(logits)
        p_t = probs * targets + (1 - probs) * (1 - targets)
        alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        focal_loss = alpha_t * (1 - p_t) ** self.gamma * bce_loss

        if self.reduction == "mean":
            return focal_loss.mean()
        if self.reduction == "sum":
            return focal_loss.sum()
        return focal_loss


def compute_pos_weight(df: pd.DataFrame, mlb, max_weight: float = 20.0) -> torch.Tensor:
    y = torch.tensor(label_matrix(df), dtype=torch.float32)
    positives = y.sum(dim=0)
    negatives = y.shape[0] - positives
    pos_weight = negatives / positives.clamp_min(1.0)
    return pos_weight.clamp(max=max_weight)


def build_criterion(loss_name: str = HannahConfig.LOSS_NAME) -> nn.Module:
    if loss_name == "weighted_bce":
        pos_weight = compute_pos_weight(train_df, mlb).to(DEVICE)
        print("Using weighted BCE. pos_weight min/median/max:",
              round(float(pos_weight.min()), 3),
              round(float(pos_weight.median()), 3),
              round(float(pos_weight.max()), 3))
        return nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    if loss_name == "focal":
        print("Using focal loss.")
        return FocalLoss(alpha=STRICT_BASELINE["focal_alpha"], gamma=STRICT_BASELINE["focal_gamma"])
    raise ValueError(f"Unknown loss_name: {loss_name}")


def compute_metrics(logits: torch.Tensor, targets: torch.Tensor, threshold: float = HannahConfig.DEFAULT_THRESHOLD) -> dict:
    probs = torch.sigmoid(logits)
    preds = (probs >= threshold).float()
    tp = (preds * targets).sum(dim=1)
    fp = (preds * (1 - targets)).sum(dim=1)
    fn = ((1 - preds) * targets).sum(dim=1)

    precision = (tp / (tp + fp + 1e-8)).mean().item()
    recall = (tp / (tp + fn + 1e-8)).mean().item()
    f1 = (2 * tp / (2 * tp + fp + fn + 1e-8)).mean().item()
    exact = (preds == targets).all(dim=1).float().mean().item()
    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "exact_match": exact,
        "pred_labels_per_sample": preds.sum(dim=1).mean().item(),
        "true_labels_per_sample": targets.sum(dim=1).mean().item(),
        "mean_prob": probs.mean().item(),
        "max_prob": probs.max().item(),
        "threshold": float(threshold),
    }


def calibrate_threshold(logits: torch.Tensor, targets: torch.Tensor, thresholds: list[float] | None = None) -> tuple[float, list[dict]]:
    thresholds = thresholds or HannahConfig.THRESHOLD_SWEEP
    rows = []
    best_threshold = thresholds[0]
    best_f1 = -1.0
    for threshold in thresholds:
        metrics = compute_metrics(logits, targets, threshold=threshold)
        row = {"threshold": threshold, **metrics}
        rows.append(row)
        if metrics["f1"] > best_f1:
            best_f1 = metrics["f1"]
            best_threshold = threshold
    return best_threshold, rows


def _autocast_enabled(device: torch.device) -> bool:
    return device.type == "cuda" and HannahConfig.USE_AMP


def train_one_epoch(model, loader, criterion, optimizer, device, scaler=None):
    model.train()
    total_loss = 0.0
    all_logits, all_targets = [], []

    for imgs, labels in loader:
        imgs = imgs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast("cuda", enabled=_autocast_enabled(device)):
            logits = model(imgs)
            loss = criterion(logits, labels)

        if scaler is not None and scaler.is_enabled():
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), HannahConfig.GRAD_CLIP_NORM)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), HannahConfig.GRAD_CLIP_NORM)
            optimizer.step()

        total_loss += loss.item() * imgs.size(0)
        all_logits.append(logits.detach().cpu())
        all_targets.append(labels.detach().cpu())

    avg_loss = total_loss / len(loader.dataset)
    metrics = compute_metrics(torch.cat(all_logits), torch.cat(all_targets))
    return avg_loss, metrics


@torch.no_grad()
def collect_logits_targets(model, loader, device) -> tuple[torch.Tensor, torch.Tensor]:
    model.eval()
    all_logits, all_targets = [], []
    for imgs, labels in loader:
        imgs = imgs.to(device, non_blocking=True)
        with torch.amp.autocast("cuda", enabled=_autocast_enabled(device)):
            logits = model(imgs)
        all_logits.append(logits.cpu())
        all_targets.append(labels.cpu())
    return torch.cat(all_logits), torch.cat(all_targets)


@torch.no_grad()
def evaluate(model, loader, criterion, device, threshold: float = HannahConfig.DEFAULT_THRESHOLD):
    model.eval()
    total_loss = 0.0
    all_logits, all_targets = [], []

    for imgs, labels in loader:
        imgs = imgs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        with torch.amp.autocast("cuda", enabled=_autocast_enabled(device)):
            logits = model(imgs)
            loss = criterion(logits, labels)
        total_loss += loss.item() * imgs.size(0)
        all_logits.append(logits.cpu())
        all_targets.append(labels.cpu())

    logits = torch.cat(all_logits)
    targets = torch.cat(all_targets)
    avg_loss = total_loss / len(loader.dataset)
    metrics = compute_metrics(logits, targets, threshold=threshold)
    return avg_loss, metrics


In [6]:
class ConvBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, dropout: float):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2),
            nn.Dropout2d(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)


class HannahScratchCNN(nn.Module):
    def __init__(self, num_labels: int, dropout: float = HannahConfig.DROPOUT):
        super().__init__()
        self.features = nn.Sequential(
            ConvBlock(3, 32, dropout=0.10),
            ConvBlock(32, 64, dropout=0.15),
            ConvBlock(64, 128, dropout=0.20),
            ConvBlock(128, 256, dropout=0.25),
        )
        self.pool = nn.AdaptiveAvgPool2d((2, 2))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 2 * 2, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(512, num_labels),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        x = self.pool(x)
        return self.classifier(x)


model = HannahScratchCNN(num_genres).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(model)
print(f"Trainable parameters: {n_params:,}")

HannahScratchCNN(
  (features): Sequential(
    (0): ConvBlock(
      (block): Sequential(
        (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU(inplace=True)
        (3): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (4): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (5): ReLU(inplace=True)
        (6): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
        (7): Dropout2d(p=0.1, inplace=False)
      )
    )
    (1): ConvBlock(
      (block): Sequential(
        (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU(inplace=True)
        (3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), paddi

In [ ]:
# Smoke test: one tiny training epoch plus validation and test evaluation.
set_seed(STRICT_BASELINE["seed"])
smoke_train_loader, smoke_val_loader, smoke_test_loader = make_loaders(
    batch_size=HannahConfig.SMOKE_BATCH_SIZE,
    smoke=True,
)
smoke_model = HannahScratchCNN(num_genres).to(DEVICE)
criterion = build_criterion()
optimizer = optim.AdamW(smoke_model.parameters(), lr=HannahConfig.LR, weight_decay=HannahConfig.WEIGHT_DECAY)
scaler = torch.amp.GradScaler("cuda", enabled=_autocast_enabled(DEVICE))

smoke_train_loss, smoke_train_m = train_one_epoch(smoke_model, smoke_train_loader, criterion, optimizer, DEVICE, scaler=scaler)
smoke_val_loss, smoke_val_m = evaluate(smoke_model, smoke_val_loader, criterion, DEVICE)
smoke_test_loss, smoke_test_m = evaluate(smoke_model, smoke_test_loader, criterion, DEVICE)

print(f"Smoke train loss: {smoke_train_loss:.4f} | train F1: {smoke_train_m['f1']:.4f} | pred labels: {smoke_train_m['pred_labels_per_sample']:.2f}")
print(f"Smoke val loss:   {smoke_val_loss:.4f} | val F1:   {smoke_val_m['f1']:.4f} | pred labels: {smoke_val_m['pred_labels_per_sample']:.2f}")
print(f"Smoke test loss:  {smoke_test_loss:.4f} | test F1:  {smoke_test_m['f1']:.4f} | pred labels: {smoke_test_m['pred_labels_per_sample']:.2f}")


In [ ]:
# Full training and final evaluation. Leave RUN_FULL_TRAINING=False for quick notebook smoke runs.
RUN_FULL_TRAINING = False
WRITE_METRICS_JSON = False
WRITE_HISTORY_JSON = False
SAVE_MODEL_CHECKPOINT = False

RESULTS_DIR = ROOT / "results" / "hannah"
CHECKPOINT_DIR = ROOT / "experimentation" / "hannah" / "checkpoints"
MODEL_CHECKPOINT_PATH = CHECKPOINT_DIR / "hannah_scratch_cnn.pt"

if RUN_FULL_TRAINING:
    set_seed(STRICT_BASELINE["seed"])
    train_loader, val_loader, test_loader = make_loaders(batch_size=HannahConfig.BATCH_SIZE, smoke=False)
    model = HannahScratchCNN(num_genres).to(DEVICE)
    criterion = build_criterion()
    optimizer = optim.AdamW(model.parameters(), lr=HannahConfig.LR, weight_decay=HannahConfig.WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=HannahConfig.LR_FACTOR,
        patience=HannahConfig.LR_PATIENCE,
        min_lr=HannahConfig.MIN_LR,
    )
    scaler = torch.amp.GradScaler("cuda", enabled=_autocast_enabled(DEVICE))

    history = []
    best_val_loss = float("inf")
    best_val_f1_at_default = -1.0
    best_state = None
    best_epoch = 0
    patience_ctr = 0

    for epoch in range(1, HannahConfig.NUM_EPOCHS + 1):
        train_loss, train_m = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE, scaler=scaler)
        val_loss, val_m = evaluate(model, val_loader, criterion, DEVICE)
        old_lr = optimizer.param_groups[0]["lr"]
        scheduler.step(val_loss)
        current_lr = optimizer.param_groups[0]["lr"]
        if current_lr < old_lr:
            print(f"Reduced LR: {old_lr:.2e} -> {current_lr:.2e}")

        row = {
            "epoch": epoch,
            "train_loss": float(train_loss),
            "val_loss": float(val_loss),
            "train_f1": float(train_m["f1"]),
            "val_f1": float(val_m["f1"]),
            "val_precision": float(val_m["precision"]),
            "val_recall": float(val_m["recall"]),
            "val_exact_match": float(val_m["exact_match"]),
            "val_pred_labels_per_sample": float(val_m["pred_labels_per_sample"]),
            "val_true_labels_per_sample": float(val_m["true_labels_per_sample"]),
            "val_mean_prob": float(val_m["mean_prob"]),
            "val_max_prob": float(val_m["max_prob"]),
            "lr": float(current_lr),
        }
        history.append(row)
        best_val_f1_at_default = max(best_val_f1_at_default, val_m["f1"])

        print(
            f"Epoch {epoch:02d}/{HannahConfig.NUM_EPOCHS} | "
            f"train loss {train_loss:.4f} | val loss {val_loss:.4f} | "
            f"val F1@{HannahConfig.DEFAULT_THRESHOLD:.2f} {val_m['f1']:.4f} | "
            f"val rec {val_m['recall']:.4f} | "
            f"pred labels {val_m['pred_labels_per_sample']:.2f}/{val_m['true_labels_per_sample']:.2f} | "
            f"lr {current_lr:.2e}"
        )

        if val_loss < (best_val_loss - HannahConfig.MIN_VAL_LOSS_DELTA):
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            best_epoch = epoch
            patience_ctr = 0
        else:
            patience_ctr += 1
            if patience_ctr >= HannahConfig.PATIENCE:
                print(f"Early stopping at epoch {epoch} based on validation loss")
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    val_logits, val_targets = collect_logits_targets(model, val_loader, DEVICE)
    best_threshold, threshold_rows = calibrate_threshold(val_logits, val_targets)
    best_threshold_metrics = compute_metrics(val_logits, val_targets, threshold=best_threshold)

    print("\nThreshold sweep on validation set:")
    print(f"{'thr':>5} | {'F1':>7} | {'prec':>7} | {'rec':>7} | {'pred/sample':>11}")
    print("-" * 48)
    for row in threshold_rows:
        marker = " <-- best" if row["threshold"] == best_threshold else ""
        print(
            f"{row['threshold']:>5.2f} | {row['f1']:>7.4f} | {row['precision']:>7.4f} | "
            f"{row['recall']:>7.4f} | {row['pred_labels_per_sample']:>11.2f}{marker}"
        )

    test_loss, test_m = evaluate(model, test_loader, criterion, DEVICE, threshold=best_threshold)
    print("\nBest epoch by val loss:", best_epoch)
    print("Best val loss:", round(float(best_val_loss), 4))
    print("Best threshold:", best_threshold)
    print("Validation metrics at best threshold:", {k: round(v, 4) for k, v in best_threshold_metrics.items() if isinstance(v, float)})
    print("Final test loss:", round(test_loss, 4))
    print("Final test metrics:", {k: round(v, 4) for k, v in test_m.items() if isinstance(v, float)})

    metrics_payload = {
        "model_name": "hannah_scratch_cnn",
        "seed": STRICT_BASELINE["seed"],
        "img_size": HannahConfig.IMG_SIZE,
        "train_size": len(train_df),
        "val_size": len(val_df),
        "test_size": len(test_df),
        "best_val_f1": float(best_threshold_metrics["f1"]),
        "test_f1": float(test_m["f1"]),
        "test_precision": float(test_m["precision"]),
        "test_recall": float(test_m["recall"]),
        "test_exact_match": float(test_m["exact_match"]),
        "num_epochs_run": len(history),
        "notes": (
            "Scratch CNN; strict cleaned splits; strict poster cache; "
            f"loss={HannahConfig.LOSS_NAME}; best_threshold={best_threshold}; "
            f"lr_scheduler=ReduceLROnPlateau(factor={HannahConfig.LR_FACTOR}, patience={HannahConfig.LR_PATIENCE}, min_lr={HannahConfig.MIN_LR}); "
            "checkpoint selected by val_loss; no pretrained backbone."
        ),
    }

    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    metrics_path = RESULTS_DIR / "hannah_scratch_cnn_metrics.json"
    history_path = RESULTS_DIR / "hannah_scratch_cnn_history.json"
    threshold_path = RESULTS_DIR / "hannah_scratch_cnn_threshold_sweep.json"

    if WRITE_METRICS_JSON:
        save_metrics_json(metrics_path, metrics_payload)
        with open(threshold_path, "w", encoding="utf-8") as f:
            json.dump(threshold_rows, f, indent=2)
            f.write("\n")
        print("Saved metrics JSON:", metrics_path)
        print("Saved threshold sweep JSON:", threshold_path)

    if WRITE_HISTORY_JSON:
        with open(history_path, "w", encoding="utf-8") as f:
            json.dump(history, f, indent=2)
            f.write("\n")
        print("Saved history JSON:", history_path)

    if SAVE_MODEL_CHECKPOINT:
        CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
        checkpoint = {
            "model_name": "hannah_scratch_cnn",
            "model_state_dict": model.state_dict(),
            "num_genres": num_genres,
            "classes": list(mlb.classes_),
            "img_size": HannahConfig.IMG_SIZE,
            "dropout": HannahConfig.DROPOUT,
            "best_val_loss": float(best_val_loss),
            "best_epoch": int(best_epoch),
            "best_threshold": float(best_threshold),
            "validation_metrics_at_best_threshold": best_threshold_metrics,
            "test_metrics": test_m,
            "history": history,
            "lr_scheduler": {
                "name": "ReduceLROnPlateau",
                "monitor": "val_loss",
                "mode": "min",
                "factor": HannahConfig.LR_FACTOR,
                "patience": HannahConfig.LR_PATIENCE,
                "min_lr": HannahConfig.MIN_LR,
            },
            "baseline": STRICT_BASELINE,
        }
        torch.save(checkpoint, MODEL_CHECKPOINT_PATH)
        print(f"Saved model checkpoint to {MODEL_CHECKPOINT_PATH}")

    drive_repo_env = os.environ.get("IEOR142B_DRIVE_REPO", "")
    if drive_repo_env:
        drive_repo = Path(drive_repo_env)
        drive_metrics_dir = drive_repo / "results" / "hannah"
        drive_ckpt_dir = drive_repo / "experimentation" / "hannah" / "checkpoints"

        if WRITE_METRICS_JSON and metrics_path.is_file():
            drive_metrics_dir.mkdir(parents=True, exist_ok=True)
            shutil.copy2(metrics_path, drive_metrics_dir / metrics_path.name)
            shutil.copy2(threshold_path, drive_metrics_dir / threshold_path.name)
            print("Synced metrics to Drive:", drive_metrics_dir / metrics_path.name)
            print("Synced threshold sweep to Drive:", drive_metrics_dir / threshold_path.name)
        if WRITE_HISTORY_JSON and history_path.is_file():
            drive_metrics_dir.mkdir(parents=True, exist_ok=True)
            shutil.copy2(history_path, drive_metrics_dir / history_path.name)
            print("Synced history to Drive:", drive_metrics_dir / history_path.name)
        if SAVE_MODEL_CHECKPOINT and MODEL_CHECKPOINT_PATH.is_file():
            drive_ckpt_dir.mkdir(parents=True, exist_ok=True)
            shutil.copy2(MODEL_CHECKPOINT_PATH, drive_ckpt_dir / MODEL_CHECKPOINT_PATH.name)
            print("Synced checkpoint to Drive:", drive_ckpt_dir / MODEL_CHECKPOINT_PATH.name)
else:
    print("Full training skipped. Set RUN_FULL_TRAINING=True to train on the full strict cleaned split.")
